# Transformer from Scratch — IMDB Sentiment Analysis

Implements all core Transformer components **manually** using TensorFlow/Keras:
- Positional Encoding
- Multi-Head Self-Attention
- Position-wise Feed-Forward Network
- Layer Normalization & Residual Connections
- Transformer Encoder Block
- Full Sentiment Classifier


## Cell 1 — Imports & Configuration

In [ ]:
import os, sys, re, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"]  = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")


## Cell 2 — Hyperparameters

In [ ]:
# ── Hyperparameters ───
VOCAB_SIZE  = 20_000   
MAX_LEN     = 200      
EMBED_DIM   = 64       
NUM_HEADS   = 4        
FF_DIM      = 128      
NUM_LAYERS  = 2        
DROPOUT     = 0.1      
BATCH_SIZE  = 64       
EPOCHS      = 5       
LR          = 1e-3     

print("Hyperparameters set.")


## Cell 3 — Load & Preprocess Data

In [ ]:
# ── Load IMDB dataset ───────────────────────────────────────
df = pd.read_csv("C:\\VSC\\IMDB Dataset.csv")   # adjust path if needed
print(f"Total samples : {len(df)}")
print(df["sentiment"].value_counts())

# Clean HTML tags, punctuation, lowercase
df["review"] = (df["review"]
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)
    .str.lower()
    .str.strip())

# Encode label: positive=1, negative=0
df["label"] = (df["sentiment"] == "positive").astype(int)

# Train / Validation / Test split  (70 / 15 / 15)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    df["review"].values, df["label"].values,
    test_size=0.30, random_state=42, stratify=df["label"])

X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp,
    test_size=0.50, random_state=42, stratify=y_tmp)

print(f"Train {len(X_tr)} | Val {len(X_va)} | Test {len(X_te)}")


## Cell 4 — Tokenization & tf.data Pipelines

In [ ]:
# ── TextVectorization (tokenizer) ───────────────────────────
vec = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN,
    standardize="lower_and_strip_punctuation")

vec.adapt(X_tr)   # Build vocabulary from training data only
print(f"Vocabulary size: {len(vec.get_vocabulary())}")

# ── Build tf.data datasets ───────────────────────────────────
def make_dataset(texts, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000, seed=42)
    return (ds
            .batch(BATCH_SIZE)
            .map(lambda x, y: (vec(x), y), num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))

tr_ds = make_dataset(X_tr, y_tr, shuffle=True)
va_ds = make_dataset(X_va, y_va)
te_ds = make_dataset(X_te, y_te)

print("Datasets ready.")


## Cell 5 — Positional Encoding

In [ ]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model

        # Build the (max_len, d_model) encoding table once
        positions = np.arange(max_len)[:, None]                     
        dims      = np.arange(d_model)[None, :]                     
        angles    = positions / np.power(10_000, (2*(dims//2)) / d_model)

        angles[:, 0::2] = np.sin(angles[:, 0::2])   # even indices → sin
        angles[:, 1::2] = np.cos(angles[:, 1::2])   # odd  indices → cos

        # Shape: (1, max_len, d_model)  — batch dimension for broadcasting
        self._pe = tf.constant(angles[np.newaxis, :, :], dtype=tf.float32)

    def call(self, x):
        # x shape: (batch, seq_len, d_model)
        return x + self._pe[:, :tf.shape(x)[1], :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"max_len": self.max_len, "d_model": self.d_model})
        return cfg

print("PositionalEncoding defined.")


## Cell 6 — Multi-Head Self-Attention

In [ ]:
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        # One linear projection matrix per Q, K, V, O
        self.Wq = layers.Dense(d_model, use_bias=False, name="Wq")
        self.Wk = layers.Dense(d_model, use_bias=False, name="Wk")
        self.Wv = layers.Dense(d_model, use_bias=False, name="Wv")
        self.Wo = layers.Dense(d_model, name="Wo")

    def _split_heads(self, x, B):
        x = tf.reshape(x, (B, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def _scaled_dot_product_attention(self, Q, K, V, mask=None):
        scale  = tf.sqrt(tf.cast(self.d_k, tf.float32))
        scores = tf.matmul(Q, K, transpose_b=True) / scale  
        if mask is not None:
            scores += mask * -1e9           # mask padding tokens
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)        
    def call(self, x, mask=None):
        B  = tf.shape(x)[0]
        Q  = self._split_heads(self.Wq(x), B)   
        K  = self._split_heads(self.Wk(x), B)
        V  = self._split_heads(self.Wv(x), B)

        # Compute attention for all heads in parallel
        attn = self._scaled_dot_product_attention(Q, K, V, mask)

        # Merge heads: (B,H,T,d_k) → (B,T,D)
        attn = tf.transpose(attn, perm=[0, 2, 1, 3])
        attn = tf.reshape(attn, (B, -1, self.d_model))
        return self.Wo(attn)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"d_model": self.d_model, "num_heads": self.num_heads})
        return cfg

print("MultiHeadSelfAttention defined.")


## Cell 7 — Position-wise Feed-Forward Network

In [ ]:
class FeedForwardNetwork(layers.Layer):
    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.fc1 = layers.Dense(d_ff,    activation="relu", name="ffn_1")
        self.fc2 = layers.Dense(d_model, name="ffn_2")

    def call(self, x):
        return self.fc2(self.fc1(x))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"d_model": self.fc2.units, "d_ff": self.fc1.units})
        return cfg

print("FeedForwardNetwork defined.")


## Cell 8 — Transformer Encoder Block

In [ ]:
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate, **kwargs):
        super().__init__(**kwargs)
        self.attn  = MultiHeadSelfAttention(d_model, num_heads, name="mhsa")
        self.ffn   = FeedForwardNetwork(d_model, d_ff, name="ffn")
        self.ln1   = layers.LayerNormalization(epsilon=1e-6)
        self.ln2   = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        # --- Sub-layer 1: Multi-Head Self-Attention ---
        attn_out = self.attn(x, mask=mask)              # attention
        attn_out = self.drop1(attn_out, training=training)
        x = self.ln1(x + attn_out)                     # residual + LN

        # --- Sub-layer 2: Feed-Forward Network ---
        ffn_out  = self.ffn(x)                         # FFN
        ffn_out  = self.drop2(ffn_out, training=training)
        x = self.ln2(x + ffn_out)                      # residual + LN
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "d_model":      self.attn.d_model,
            "num_heads":    self.attn.num_heads,
            "d_ff":         self.ffn.fc1.units,
            "dropout_rate": self.drop1.rate,
        })
        return cfg

print("TransformerEncoderBlock defined.")


## Cell 9 — Full Transformer Classifier

In [ ]:
class TransformerClassifier(keras.Model):
    def __init__(self, vocab_size, max_len, d_model,
                 num_heads, d_ff, num_layers, dropout_rate, **kwargs):
        super().__init__(**kwargs)
        self.embedding  = layers.Embedding(vocab_size, d_model)
        self.pos_enc    = PositionalEncoding(max_len, d_model)
        self.emb_drop   = layers.Dropout(dropout_rate)
        self.enc_blocks = [
            TransformerEncoderBlock(d_model, num_heads, d_ff,
                                    dropout_rate, name=f"encoder_block_{i}")
            for i in range(num_layers)
        ]
        self.pool       = layers.GlobalAveragePooling1D()
        self.final_drop = layers.Dropout(dropout_rate)
        self.dense      = layers.Dense(64, activation="relu")
        self.output_    = layers.Dense(1,  activation="sigmoid", name="output")

    def call(self, x, training=False):
        mask = tf.cast(tf.equal(x, 0), tf.float32)[:, tf.newaxis, tf.newaxis, :]

        x = self.embedding(x)
        x = self.pos_enc(x)                                  
        x = self.emb_drop(x, training=training)

        for block in self.enc_blocks:
            x = block(x, training=training, mask=mask)      

        x = self.pool(x)                                     
        x = self.final_drop(x, training=training)
        x = self.dense(x)
        return self.output_(x)                              

print("TransformerClassifier defined.")


## Cell 10 — Build & Compile Model

In [ ]:
model = TransformerClassifier(
    vocab_size    = VOCAB_SIZE,
    max_len       = MAX_LEN,
    d_model       = EMBED_DIM,
    num_heads     = NUM_HEADS,
    d_ff          = FF_DIM,
    num_layers    = NUM_LAYERS,
    dropout_rate  = DROPOUT,
    name          = "transformer_from_scratch"
)

# Warm-up call to build all weights
_ = model(tf.zeros((1, MAX_LEN), dtype=tf.int32), training=False)
model.summary()

print(f"Total trainable parameters: {model.count_params():,}")


## Cell 11 — Compile

In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LR),
    loss      = "binary_crossentropy",
    metrics   = [
        "accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ]
)

print("Model compiled.")


## Cell 12 — Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
        verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1),
]

history = model.fit(
    tr_ds,
    validation_data = va_ds,
    epochs          = EPOCHS,
    callbacks       = callbacks,
    verbose         = 1,
)


## Cell 13 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history["accuracy"],     label="Train Accuracy",     linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2, linestyle="--")
axes[0].set_title("Accuracy over Epochs",  fontsize=14)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history["loss"],     label="Train Loss",     linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
axes[1].set_title("Loss over Epochs",  fontsize=14)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 14 — Evaluate on Test Set

In [ ]:
results = model.evaluate(te_ds, verbose=1)
metric_names = ["loss", "accuracy", "auc", "precision", "recall"]
metrics = dict(zip(metric_names, results))

f1 = (2 * metrics["precision"] * metrics["recall"] / (metrics["precision"] + metrics["recall"] + 1e-8))

print("" + "="*40)
print(f"  Loss      : {metrics['loss']:.4f}")
print(f"  Accuracy  : {metrics['accuracy']:.4f}")
print(f"  AUC       : {metrics['auc']:.4f}")
print(f"  Precision : {metrics['precision']:.4f}")
print(f"  Recall    : {metrics['recall']:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("="*40)


## Cell 15 — Per-Class Accuracy & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Collect all predictions
all_preds, all_labels = [], []
for batch_x, batch_y in te_ds:
    probs = model(batch_x, training=False).numpy().ravel()
    all_preds.extend((probs >= 0.5).astype(int))
    all_labels.extend(batch_y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

pos_acc = (all_preds[all_labels == 1] == 1).mean()
neg_acc = (all_preds[all_labels == 0] == 0).mean()
print(f"Positive-class accuracy : {pos_acc:.4f}")
print(f"Negative-class accuracy : {neg_acc:.4f}")

print("Classification Report:")
print(classification_report(all_labels, all_preds,
                             target_names=["Negative", "Positive"]))

# Confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 16 — Inference on New Reviews

In [ ]:
demo_reviews = [
    "This movie was absolutely brilliant! A masterpiece of storytelling.",
    "Terrible waste of time. Boring, predictable and poorly acted.",
    "The film had some good moments but overall felt very average.",
    "One of the best films I have ever seen — incredible performances!",
    "I walked out halfway through. Worst cinema experience ever.",
    "A visually stunning and emotionally powerful experience.",
]

tokens = vec(tf.constant(demo_reviews))
probs  = model(tokens, training=False).numpy().ravel()

print(f"{'Review':<60}  {'Score':>6}  Label")
print("-" * 80)
for review, score in zip(demo_reviews, probs):
    label = "POSITIVE" if score >= 0.5 else "NEGATIVE"
    print(f"{review[:58]:<60}  {score:.4f}  {label}")


## Cell 17 — Training History Table

In [ ]:
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc'}")
print("  " + "-" * 52)
for i, (tl, ta, vl, va) in enumerate(zip(
        history.history["loss"],
        history.history["accuracy"],
        history.history["val_loss"],
        history.history["val_accuracy"]), 1):
    print(f"  {i:<6} {tl:<12.4f} {ta:<12.4f} {vl:<12.4f} {va:.4f}")
